# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mariamemad975/FlyRank_ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib


In [2]:
!pwd
!ls

/content
sample_data


In [3]:
!git clone https://github.com/mariamemad975/FlyRank_ML.git
%cd FlyRank_ML

Cloning into 'FlyRank_ML'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 164 (delta 70), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 1.87 MiB | 9.06 MiB/s, done.
Resolving deltas: 100% (70/70), done.
/content/FlyRank_ML


In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("shape:", df.shape)
print(df.columns.tolist())

shape: (30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [8]:
col_ctr = "ctr"
col_position = "avg_position"
col_staleness = "days_since_last_update"
col_volume = None
col_trend_dir = "trend_direction"


df["is_declining_label"] = (
    df[col_trend_dir].astype(str).str.lower() == "down"
).astype(int)

print("CTR column:       ", col_ctr)
print("Position column:  ", col_position)
print("Staleness column: ", col_staleness)
print("Volume column:    ", col_volume)
print("Trend/label col:  ", col_trend_dir)

CTR column:        ctr
Position column:   avg_position
Staleness column:  days_since_last_update
Volume column:     None
Trend/label col:   trend_direction


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [11]:
position_bins = [0, 3, 10, 20, 50, np.inf]
position_labels = ["1-3", "4-10", "11-20", "21-50", "50+"]

df["position_bucket"] = pd.cut(
    df[col_position],
    bins=position_bins,
    labels=position_labels)

signal_a = (
    df.groupby("position_bucket", observed=True)
      .agg(
          mean_ctr=(col_ctr, "mean"),
          n=(col_ctr, "size"))
      .reset_index())
display(signal_a)

is_monotonic_decreasing = signal_a["mean_ctr"].is_monotonic_decreasing
print(
    "\nVerdict:",
    "CONFIRMED" if is_monotonic_decreasing else "MIXED",
    "— CTR",
    "decreases" if is_monotonic_decreasing else "does not consistently decrease",
    "as search position worsens.")
print("Total pages:", signal_a["n"].sum())

,position_bucket,mean_ctr,n
0,1-3,2.714303,1141
1,4-10,0.651045,11842
2,11-20,0.323443,7273
3,21-50,0.222345,7225
4,50+,0.150784,1314



Verdict: CONFIRMED — CTR decreases as search position worsens.
Total pages: 28795


In [13]:
stale_bins = [0, 30, 90, 180, 365, np.inf]
stale_labels = ["0-30d", "31-90d", "91-180d", "181-365d", "365d+"]

df["staleness_bucket"] = pd.cut(
    df[col_staleness],
    bins=stale_bins,
    labels=stale_labels)

signal_b = (
    df.groupby("staleness_bucket", observed=True)
      .agg(
          decline_rate=("is_declining_label", "mean"),
          n=("is_declining_label", "size"))
      .reset_index())
display(signal_b)

is_increasing = signal_b["decline_rate"].is_monotonic_increasing
print(
    "\nVerdict:",
    "CONFIRMED" if is_increasing else "MIXED",
    "— decline rate",
    "rises" if is_increasing else "does not consistently rise",
    "as content becomes older.")
print("Total pages:", signal_b["n"].sum())

,staleness_bucket,decline_rate,n
0,0-30d,0.511377,20480
1,31-90d,0.588571,175
2,91-180d,0.611057,9171
3,181-365d,0.467456,169
4,365d+,0.600000,5



Verdict: MIXED — decline rate does not consistently rise as content becomes older.
Total pages: 30000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
col_volume = "impressions_last_30d"
expected_ctr = (
    df.groupby("position_bucket", observed=True)[col_ctr]
      .transform("mean"))
df["ctr_gap"] = expected_ctr - df[col_ctr]
MIN_IMPRESSIONS = 30

def calculate_score(row):
    if row[col_volume] < MIN_IMPRESSIONS:
        return pd.Series({
            "action_score": 0.0,
            "reason_code": "INSUFFICIENT_VOLUME"})
    score = 0.0
    reason = "LOW_SIGNAL"

    if row["ctr_gap"] > 0:
        score += row["ctr_gap"] * 100
        reason = "CTR_BELOW_EXPECTED_FOR_POSITION"

    if pd.notna(row[col_staleness]):
        if row[col_staleness] > 180 and row[col_position] <= 20:
            score += 10
            if reason == "LOW_SIGNAL":
                reason = "STALE_HIGH_POSITION"
    return pd.Series({
        "action_score": score,
        "reason_code": reason})

df[["action_score", "reason_code"]] = df.apply(
    calculate_score,
    axis=1)

df["action_score_tiebreak"] = (
    df["action_score"]
    + (df[col_volume] / df[col_volume].max()) * 0.01)

def action_label(score):
    if score >= 15:
        return "Refresh Now"
    elif score >= 5:
        return "Monitor"
    else:
        return "No Action"

df["action_label"] = df["action_score"].apply(action_label)

queue = (
    df.sort_values("action_score_tiebreak", ascending=False)
      .reset_index(drop=True))

os.makedirs("work/outputs", exist_ok=True)
output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)
print("Wrote",len(queue),"ranked rows to",output_path)

display(queue[[col_position,
            col_ctr,
            col_volume,
            "action_score",
            "reason_code",
            "action_label"]].head(10))

Wrote 30000 ranked rows to work/outputs/baseline_action_score.csv


,avg_position,ctr,impressions_last_30d,action_score,reason_code,action_label
0,2.4,0.0,1444,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
1,1.5,0.0,303,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
2,2.7,0.0,245,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
3,1.6,0.0,232,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
4,1.8,0.0,212,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
5,1.8,0.0,197,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
6,2.6,0.0,184,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
7,2.3,0.0,172,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
8,2.8,0.0,139,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now
9,2.7,0.0,136,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()
top20["confidence_note"] = np.where(top20["action_score"] >= 20, "high", "moderate")
top20["what_would_make_it_wrong"] = ""

top20[[col_position, col_ctr, "action_score", "reason_code", "action_label",
       "confidence_note", "what_would_make_it_wrong"]]

,avg_position,ctr,action_score,reason_code,action_label,confidence_note,what_would_make_it_wrong
0,2.4,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
1,1.5,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
2,2.7,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
3,1.6,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
4,1.8,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
5,1.8,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
6,2.6,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
7,2.3,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
8,2.8,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,
9,2.7,0.0,271.430324,CTR_BELOW_EXPECTED_FOR_POSITION,Refresh Now,high,


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_inputs_used = [col_ctr,
    col_position,
    col_staleness]

leak_terms = [
    "trend_direction",
    "trend_pct"]

leaked = [col for col in feature_inputs_used
    if col and any(term in col.lower() for term in leak_terms)]
assert not leaked, f"Leakage detected: {leaked}"

print("Leak check passed — no label-derived inputs are used in scoring.")

median_score = top20["action_score"].median()
weak_candidates = top20[
    top20["action_score"] < median_score]

print(f"\n{len(weak_candidates)} of the top 20 "
    f"have scores below the group's median.")
print("Review these first for potential false positives:")

display(weak_candidates[[col_position,
            col_ctr,
            "action_score",
            "reason_code"]])

Leak check passed — no label-derived inputs are used in scoring.

0 of the top 20 have scores below the group's median.
Review these first for potential false positives:


,avg_position,ctr,action_score,reason_code


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.